In [ ]:
import sys
!{sys.executable} -m pip install -q pydantic-ai-slim openai mcp-server-time "fastmcp-slim[client]"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("groq-api-key")

os.environ["GROQ_API_KEY"]   = secret_value_0
os.environ["BASE_URL"]       = "https://api.groq.com/openai/v1"
os.environ["OPENAI_API_KEY"] = secret_value_0

print("Config ready. BASE_URL:", os.environ["BASE_URL"])

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

models = client.models.list()
print("Connected! Available models:")
for m in models.data[:5]:
    print(" •", m.id)

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("llama-3.3-70b-versatile", provider=provider)

print("Agent model ready!")

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

time_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/New_York"],
    )
)

agent = Agent(
    model=agent_model,
    toolsets=[time_server],
    system_prompt=(
        "You MUST use the get_current_time tool to answer any question about the current date or time. "
        "Never say you don't have access to the time. Always call the tool first, then respond."
    )
)

print("Agent with MCP time server ready!")

In [ ]:
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output

In [ ]:
answer = await run_async("What's the date today?")
print(answer)

In [ ]:
import sys
!{sys.executable} -m pip install -q mcp-server-fetch

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

time_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/New_York"],
    )
)

fetch_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_fetch"],
    )
)

agent = Agent(
    model=agent_model,
    toolsets=[time_server, fetch_server],
    system_prompt=(
        "You are a helpful agent with access to two tools: "
        "get_current_time for date/time questions, and fetch for retrieving web content. "
        "Always use the appropriate tool when needed."
    )
)

print("Agent with time + fetch MCP servers ready!")

In [ ]:
answer = await run_async("Fetch the content from https://example.com and summarize it.")
print(answer)

In [ ]:
answer = await run_async(
    "What is today's date and time? Also fetch https://example.com and give me a one line summary."
)
print(answer)